In [ ]:
import pandas as pd
import numpy as np
import random
from typing import Set, Tuple, Dict, Any, List
from pathlib import Path
from tqdm import tqdm
import networkx as nx
import matplotlib.pyplot as plt
plt.style.use("../rw_visualization.mplstyle")

In [ ]:
pkn_file = Path("../data/corrected_PKN_with_BRAF_fixes.csv")
# pkn_file = Path("../data/clean_omnipath_PKN.csv")
pkn = pd.read_csv(pkn_file)

In [ ]:
pkn.head()

In [ ]:
def pd_to_nx(pkn: pd.DataFrame) -> nx.DiGraph:
    """Convert a pandas DataFrame to a NetworkX directed graph.

    Args:
        pkn (pd.DataFrame): DataFrame containing the data with columns 'source', 'target', and 'interaction'.

    Returns:
        nx.DiGraph: A directed graph representing the interactions.
    """
    G = nx.DiGraph()
    
    for _, row in pkn.iterrows():
        source = row['source']
        target = row['target']
        interaction = row['interaction']
        
        # Add nodes if they don't exist
        if source not in G:
            G.add_node(source)
        if target not in G:
            G.add_node(target)
        
        # Add edge with interaction attribute
        G.add_edge(source, target, interaction=interaction)
    
    return G

In [ ]:
G = pd_to_nx(pkn)
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")

# Modularity analysis 

In [ ]:
communities_len = []
modularities = []

for seed in range(100):
    communities = nx.community.louvain_communities(G, seed=seed)
    modularity = nx.community.modularity(G, communities)
    communities_len.append(len(communities))
    modularities.append(modularity)

In [ ]:
# comunity_sizes = []
# for community in communities:
#     comunity_sizes.append(len(community))

In [ ]:
plt.figure()
plt.plot(communities_len, modularities, '.')
plt.xlabel("Number of Louvain communities")
plt.ylabel("Modularity")
print(f"Mean modularity: {np.mean(modularities)}")

In [ ]:
# communities = nx.community.girvan_newman(G)
# modularity = nx.community.modularity(G, communities)
# print(f"Girvan-Newman modularity: {modularity}")

In [ ]:
# len(nx.community.label_propagation_communities(G.to_undirected()))

# Assortativity analysis

In [ ]:
r = nx.degree_assortativity_coefficient(G)
print(f"{r:3.1f}")

# Subgraph density

In [ ]:
subgraph_density, subgraph_nodes = nx.approximation.densest_subgraph(G.to_undirected(), iterations=100, method='greedy++')

In [ ]:
print(f"Densest subgraph density: {subgraph_density}")
print(len(subgraph_nodes))

In [ ]:
densest_subgraph = G.subgraph(subgraph_nodes)

In [ ]:
print(f"Number of nodes in densest subgraph: {densest_subgraph.number_of_nodes()}")

In [ ]:
nx.density(densest_subgraph.to_undirected())

In [ ]:
nx.density(G.to_undirected())

In [ ]:
# Check the densities for all nodes except the densest subgraph
all_nodes = set(G.nodes())
remaining_nodes = all_nodes - set(subgraph_nodes)
remaining_subgraph = G.subgraph(remaining_nodes)
nx.density(remaining_subgraph.to_undirected())

In [ ]:
# Get the subgraphs by communities
subgraphs = [G.subgraph(nodes) for nodes in communities]

In [ ]:
# Calculate density for each subgraph
subgraph_densities = [nx.density(sg) for sg in subgraphs]
plt.hist(subgraph_densities, bins=100)
plt.xlabel("Density")
plt.ylabel("Frequency")
plt.show()
print(f"Mean subgraph density: {np.mean(subgraph_densities)}")
print(f"Max subgraph density: {np.max(subgraph_densities)}")
print(f"Min subgraph density: {np.min(subgraph_densities)}")

In [ ]:
subgraph_num_nodes = [sg.number_of_nodes() for sg in subgraphs]
plt.hist(subgraph_num_nodes, bins=100)
plt.xlabel("Number of nodes")
plt.ylabel("Frequency")
print(f"Mean number of nodes: {np.mean(subgraph_num_nodes)}")
print(f"Max number of nodes: {np.max(subgraph_num_nodes)}")
print(f"Min number of nodes: {np.min(subgraph_num_nodes)}")

In [ ]:
plt.figure()
plt.plot(subgraph_num_nodes, subgraph_densities, '.')
plt.xscale("log")
plt.xlabel("Number of nodes")
plt.ylabel("Density")